In [26]:
!pip uninstall gsplat -y

Found existing installation: gsplat 1.5.2
Uninstalling gsplat-1.5.2:
  Successfully uninstalled gsplat-1.5.2


In [27]:
!pip install git+https://github.com/nerfstudio-project/gsplat.git

  Cloning https://github.com/nerfstudio-project/gsplat.git to /tmp/pip-req-build-ozd8y2bz
  Running command git clone --filter=blob:none --quiet https://github.com/nerfstudio-project/gsplat.git /tmp/pip-req-build-ozd8y2bz
  Resolved https://github.com/nerfstudio-project/gsplat.git to commit 0b4dddf04cb687367602c01196913cde6a743d70
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [2879 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib.linux-x86_64-cpython-312
      creating build/lib.linux-x86_64-cpython-312/gsplat
      copying gsplat/profile.py -> build/lib.linux-x86_64-cpython-312/gsplat
      copying gsplat/__init__.py -> build/lib.linux-x86_64-cpython-312/gsplat
      copying gsplat/relocation.py -> build/lib.linux-x86_64-cpython

In [3]:
!pip show gsplat

Name: gsplat
Version: 1.5.2
Summary: Python package for differentiable rasterization of gaussians
Home-page: https://github.com/nerfstudio-project/gsplat
Author: 
Author-email: 
License: 
Location: /opt/conda/lib/python3.12/site-packages
Requires: jaxtyping, ninja, numpy, rich, torch
Required-by: 


In [4]:
# Cell 1: Path declaration and setup
import uuid
from pathlib import Path
import shutil
import numpy as np
import torch
import gsplat
from PIL import Image
import struct

# User options from frontend
USE_DOWNSAMPLE = False
USE_REMBG = True

# Video file path (e.g. uploaded from frontend)
video_path = Path("/home/jovyan/mira3d/shiva.MOV")

# Setup UUID run directory
run_uuid = str(uuid.uuid4())
run_dir = Path("../data") / run_uuid
frames_dir = run_dir / "frames"
masked_dir = run_dir / "masked_frames"
colmap_dir = run_dir / "colmap"
gsplat_dir = run_dir / "gsplat" 

# Create directories
run_dir.mkdir(parents=True, exist_ok=True)
frames_dir.mkdir(exist_ok=True)
if USE_REMBG:
    masked_dir.mkdir(exist_ok=True)
colmap_dir.mkdir(exist_ok=True)
gsplat_dir.mkdir(exist_ok=True)

# Copy video to run folder
input_video_path = run_dir / "input.mp4"
shutil.copy(video_path, input_video_path)
print(" Setup done:", run_uuid)

 Setup done: 555ea092-7eb5-4a6b-a3a2-f720309fdda2


In [5]:
# Cell 2: Extract frames from video using ffmpeg with cuda acceleration
import subprocess

fps = "2"
subprocess.run([
    "ffmpeg", "-hwaccel", "cuda", "-i", str(input_video_path),
    "-vf", f"fps={fps}",
    str(frames_dir / "%04d.jpg")
], check=True)
print(" Frames extracted:", len(list(frames_dir.glob('*.jpg'))))


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

 Frames extracted: 116


In [6]:
# Cell 3: Optional downsampling with ImageMagick
if USE_DOWNSAMPLE:
    downsample_dir = run_dir / "frames_2"
    downsample_dir.mkdir(exist_ok=True)
    # Resize all images to 50%
    subprocess.run([
        "magick", "mogrify", "-path", str(downsample_dir), 
        "-resize", "50%", f"{frames_dir}/*.jpg"
    ], check=True)
    # Use downsampled frames for rest of pipeline
    frames_dir = downsample_dir
    print(" Images downsampled to 50%.")

In [7]:
# Cell 4: Background removal using ONNX model (BirefNet)
if USE_REMBG:
    import onnxruntime as ort
    import numpy as np
    from PIL import Image
    from torchvision import transforms
    from tqdm import tqdm
    import gc
    
    # Load ONNX model and configure GPU execution
    onnx_model_path = "/home/jovyan/datafabric/birefnetgeneral-model/birefnet-general.onnx"
    session = ort.InferenceSession(onnx_model_path, providers=["CUDAExecutionProvider"])
    input_name = session.get_inputs()[0].name
    
    # Preprocessing pipeline
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -250, 250)))
    
    print("Removing backgrounds... please wait")
    for img_file in tqdm(frames_dir.glob("*.jpg"), desc="Processing frames"):
        # Load and preprocess
        img = Image.open(img_file).convert("RGB")
        w, h = img.size
        input_tensor = transform(img).unsqueeze(0).numpy().astype(np.float32)
        
        # Run inference
        pred = session.run(None, {input_name: input_tensor})[0].squeeze()
        mask = Image.fromarray((sigmoid(pred) * 255).astype(np.uint8)).resize((w, h))
        
        # Apply alpha mask
        mask_arr = np.array(mask) / 255.0
        result_img = img.copy()
        result_img.putalpha(Image.fromarray((mask_arr * 255).astype(np.uint8)))
        
        # Save masked image as PNG
        result_img.save(masked_dir / f"{img_file.stem}.png")
    
    # Use masked frames for the rest of the pipeline
    frames_dir = masked_dir
    print(" Backgrounds removed using birefnet model.")
    
    # FREE GPU MEMORY USED BY ONNX
    del session, onnx_model_path, input_tensor, pred, result_img, mask_arr, mask
    gc.collect()
    torch.cuda.empty_cache()

Removing backgrounds... please wait


/tmp/ipykernel_26919/399180933.py:23: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-np.clip(x, -250, 250)))
Processing frames: 116it [00:49,  2.35it/s]

 Backgrounds removed using birefnet model.


In [8]:
# Cell 5: COLMAP Feature extraction
DB_PATH = colmap_dir / "database.db"
subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "PINHOLE",
    "--SiftExtraction.use_gpu", "1"
], check=True)
print("✅ COLMAP feature extraction complete.")

I0617 01:36:52.604001 27564 misc.cc:44] 
Feature extraction
I0617 01:36:52.604959 27577 sift.cc:721] Creating SIFT GPU feature extractor
I0617 01:36:52.764232 27578 feature_extraction.cc:259] Processed file [1/116]
I0617 01:36:52.764252 27578 feature_extraction.cc:262]   Name:            0001.png
I0617 01:36:52.764254 27578 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0617 01:36:52.764256 27578 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0617 01:36:52.764261 27578 feature_extraction.cc:277]   Focal Length:    2304.00px
I0617 01:36:52.764267 27578 feature_extraction.cc:281]   Features:        2168
I0617 01:36:53.172976 27578 feature_extraction.cc:259] Processed file [2/116]
I0617 01:36:53.173013 27578 feature_extraction.cc:262]   Name:            0002.png
I0617 01:36:53.173017 27578 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0617 01:36:53.173018 27578 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0617 01:36:53.173023 27578 feat

✅ COLMAP feature extraction complete.


I0617 01:36:56.253331 27578 feature_extraction.cc:259] Processed file [115/116]
I0617 01:36:56.253382 27578 feature_extraction.cc:262]   Name:            0115.png
I0617 01:36:56.253384 27578 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0617 01:36:56.253386 27578 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0617 01:36:56.253389 27578 feature_extraction.cc:277]   Focal Length:    2304.00px
I0617 01:36:56.253396 27578 feature_extraction.cc:281]   Features:        1293
I0617 01:36:56.269073 27578 feature_extraction.cc:259] Processed file [116/116]
I0617 01:36:56.269080 27578 feature_extraction.cc:262]   Name:            0116.png
I0617 01:36:56.269083 27578 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0617 01:36:56.269084 27578 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0617 01:36:56.269086 27578 feature_extraction.cc:277]   Focal Length:    2304.00px
I0617 01:36:56.269090 27578 feature_extraction.cc:281]   Features:        1428
I0

In [9]:
# Cell 6: COLMAP Matching
subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", str(DB_PATH),
    "--SiftMatching.use_gpu", "1"
], check=True)
print("✅ COLMAP matching complete.")

I0617 01:37:08.021160 27641 misc.cc:44] 
Feature matching
I0617 01:37:08.021705 27642 sift.cc:1428] Creating SIFT GPU feature matcher
I0617 01:37:08.150928 27641 pairing.cc:172] Generating exhaustive image pairs...
I0617 01:37:08.150942 27641 pairing.cc:205] Matching block [1/3, 1/3]
I0617 01:37:08.394462 27641 feature_matching.cc:47] in 0.244s
I0617 01:37:08.394490 27641 pairing.cc:205] Matching block [1/3, 2/3]
I0617 01:37:08.574680 27641 feature_matching.cc:47] in 0.180s
I0617 01:37:08.574702 27641 pairing.cc:205] Matching block [1/3, 3/3]
I0617 01:37:08.604570 27641 feature_matching.cc:47] in 0.030s
I0617 01:37:08.604589 27641 pairing.cc:205] Matching block [2/3, 1/3]
I0617 01:37:08.804078 27641 feature_matching.cc:47] in 0.199s
I0617 01:37:08.804100 27641 pairing.cc:205] Matching block [2/3, 2/3]
I0617 01:37:08.947563 27641 feature_matching.cc:47] in 0.143s
I0617 01:37:08.947582 27641 pairing.cc:205] Matching block [2/3, 3/3]
I0617 01:37:08.963191 27641 feature_matching.cc:47] in 

✅ COLMAP matching complete.


I0617 01:37:09.149253 27641 feature_matching.cc:47] in 0.090s
I0617 01:37:09.149271 27641 pairing.cc:205] Matching block [3/3, 3/3]
I0617 01:37:09.170334 27641 feature_matching.cc:47] in 0.021s
I0617 01:37:09.170351 27641 timer.cc:91] Elapsed time: 0.019 [minutes]


In [10]:
# Cell 7: COLMAP Mapping (Sparse reconstruction)
SPARSE_PATH = colmap_dir / "sparse"
SPARSE_PATH.mkdir(exist_ok=True)
subprocess.run([
    "colmap", "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--output_path", str(SPARSE_PATH)
], check=True)
print("✅ COLMAP sparse reconstruction complete.")

I0617 01:37:11.863140 27673 incremental_pipeline.cc:253] Loading database
I0617 01:37:11.864411 27673 database_cache.cc:66] Loading rigs...
I0617 01:37:11.864442 27673 database_cache.cc:76]  1 in 0.000s
I0617 01:37:11.864446 27673 database_cache.cc:84] Loading cameras...
I0617 01:37:11.864455 27673 database_cache.cc:102]  1 in 0.000s
I0617 01:37:11.864457 27673 database_cache.cc:110] Loading frames...
I0617 01:37:11.864559 27673 database_cache.cc:127]  116 in 0.000s
I0617 01:37:11.864563 27673 database_cache.cc:135] Loading matches...
I0617 01:37:11.866463 27673 database_cache.cc:140]  792 in 0.002s
I0617 01:37:11.866477 27673 database_cache.cc:156] Loading images...
I0617 01:37:11.871526 27673 database_cache.cc:238]  116 in 0.005s (connected 116)
I0617 01:37:11.871536 27673 database_cache.cc:249] Building correspondence graph...
I0617 01:37:11.879974 27673 database_cache.cc:276]  in 0.008s (ignored 0)
I0617 01:37:11.880060 27673 timer.cc:91] Elapsed time: 0.000 [minutes]
I0617 01:37:1

✅ COLMAP sparse reconstruction complete.


I0617 01:37:31.815296 27673 incremental_pipeline.cc:571] Discarding reconstruction due to insufficient size
I0617 01:37:31.816583 27673 incremental_pipeline.cc:299] Finding good initial image pair
I0617 01:37:31.818221 27673 incremental_pipeline.cc:323] Registering initial image pair #90 and #89
I0617 01:37:31.818286 27673 incremental_pipeline.cc:337] Global bundle adjustment
I0617 01:37:31.867661 27673 incremental_pipeline.cc:571] Discarding reconstruction due to insufficient size
I0617 01:37:31.869001 27673 incremental_pipeline.cc:299] Finding good initial image pair
I0617 01:37:31.872249 27673 incremental_pipeline.cc:323] Registering initial image pair #17 and #16
I0617 01:37:31.872303 27673 incremental_pipeline.cc:337] Global bundle adjustment
I0617 01:37:31.921650 27673 incremental_pipeline.cc:571] Discarding reconstruction due to insufficient size
I0617 01:37:31.923089 27673 incremental_pipeline.cc:299] Finding good initial image pair
I0617 01:37:31.924377 27673 incremental_pipel

In [11]:
# Cell 8: Copy images into sparse/0 and convert COLMAP binary to text
image_output_dir = SPARSE_PATH / "0"
for img in frames_dir.glob("*.[jp][pn]g"):
    shutil.copy(img, image_output_dir)

# Convert COLMAP binary files to text format for easier reading
print("🔄 Converting COLMAP binary files to text format...")
subprocess.run([
    "colmap", "model_converter",
    "--input_path", str(image_output_dir),
    "--output_path", str(image_output_dir),
    "--output_type", "TXT"
], check=True)
print(" COLMAP binary to text conversion complete.")

🔄 Converting COLMAP binary files to text format...
✅ COLMAP binary to text conversion complete.


In [23]:
# Cell 9: COLMAP Data Loader for GSplat (Fixed Matrix Formats)
def read_colmap_cameras(cameras_file):
    """Read COLMAP cameras.txt file."""
    cameras = {}
    with open(cameras_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            camera_id = int(parts[0])
            model = parts[1]
            width = int(parts[2])
            height = int(parts[3])
            params = [float(x) for x in parts[4:]]
            cameras[camera_id] = {
                'model': model, 'width': width, 'height': height, 'params': params
            }
    return cameras

def read_colmap_images(images_file):
    """Read COLMAP images.txt file."""
    images = {}
    with open(images_file, 'r') as f:
        lines = f.readlines()
        for i in range(0, len(lines), 2):  # Every other line
            line = lines[i]
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            image_id = int(parts[0])
            qw, qx, qy, qz = [float(x) for x in parts[1:5]]
            tx, ty, tz = [float(x) for x in parts[5:8]]
            camera_id = int(parts[8])
            name = parts[9]
            images[image_id] = {
                'quat': np.array([qw, qx, qy, qz]),
                'trans': np.array([tx, ty, tz]),
                'camera_id': camera_id,
                'name': name
            }
    return images

def read_colmap_points3d(points3d_file):
    """Read COLMAP points3D.txt file."""
    points = []
    with open(points3d_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            x, y, z = [float(parts[i]) for i in [1, 2, 3]]
            r, g, b = [int(parts[i]) for i in [4, 5, 6]]
            points.append([x, y, z, r/255.0, g/255.0, b/255.0])
    return np.array(points)

def quat_to_rotation_matrix(quat):
    """Convert quaternion to rotation matrix."""
    qw, qx, qy, qz = quat
    R = np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw), 2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)]
    ])
    return R

def create_view_matrix(R, t):
    """Create view matrix from rotation and translation."""
    # COLMAP uses camera-to-world, we need world-to-camera
    R_inv = R.T
    t_new = -R_inv @ t
    
    view_matrix = torch.eye(4, dtype=torch.float32)
    view_matrix[:3, :3] = torch.tensor(R_inv, dtype=torch.float32)
    view_matrix[:3, 3] = torch.tensor(t_new, dtype=torch.float32)
    return view_matrix

def load_colmap_data(colmap_path, image_dir):
    """Load COLMAP data and convert to format needed by gsplat."""
    
    # Check if files exist
    cameras_file = colmap_path / "cameras.txt"
    images_file = colmap_path / "images.txt"
    points3d_file = colmap_path / "points3D.txt"
    
    if not cameras_file.exists():
        print(f" Cannot find {cameras_file}")
        print(f" Files in {colmap_path}:")
        for f in colmap_path.iterdir():
            print(f"  - {f.name}")
        raise FileNotFoundError(f"COLMAP text files not found in {colmap_path}")
    
    cameras = read_colmap_cameras(cameras_file)
    images = read_colmap_images(images_file)
    points3d = read_colmap_points3d(points3d_file)
    
    image_data = []
    viewmats = []
    Ks = []  # Intrinsic matrices instead of projection matrices
    
    for img_id, img_info in images.items():
        img_path = image_dir / img_info['name']
        if img_path.exists():
            img = Image.open(img_path).convert('RGB')
            img_tensor = torch.tensor(np.array(img), dtype=torch.float32) / 255.0
            image_data.append(img_tensor)
            
            # Get camera info
            cam_info = cameras[img_info['camera_id']]
            width, height = cam_info['width'], cam_info['height']
            
            # Create view matrix
            R = quat_to_rotation_matrix(img_info['quat'])
            t = img_info['trans']
            viewmat = create_view_matrix(R, t)
            viewmats.append(viewmat)
            
            # Create intrinsic matrix K
            if cam_info['model'] == 'PINHOLE':
                fx, fy, cx, cy = cam_info['params']
                K = torch.tensor([
                    [fx, 0, cx],
                    [0, fy, cy],
                    [0, 0, 1]
                ], dtype=torch.float32)
                Ks.append(K)
    
    return {
        'images': image_data,
        'viewmats': viewmats,
        'Ks': Ks,  # Changed from projmats to Ks
        'points3d': points3d,
        'width': width,
        'height': height
    }

# Load COLMAP data
colmap_model_path = SPARSE_PATH / "0"
print("📊 Loading COLMAP data...")
colmap_data = load_colmap_data(colmap_model_path, image_output_dir)
print(f" Loaded {len(colmap_data['images'])} images and {len(colmap_data['points3d'])} 3D points")

📊 Loading COLMAP data...
✅ Loaded 67 images and 5025 3D points


In [24]:
# Cell 10: Initialize Gaussians from COLMAP points (Fixed In-Place Operations)
def initialize_gaussians(points3d, device='cuda'):
    """Initialize Gaussian parameters from COLMAP 3D points."""
    num_points = len(points3d)
    
    # 3D positions from COLMAP points
    positions = torch.tensor(points3d[:, :3], dtype=torch.float32, device=device, requires_grad=True)
    
    # Initialize scales (small initial values)
    scales = torch.full((num_points, 3), 0.01, dtype=torch.float32, device=device, requires_grad=True)
    
    # Initialize rotations as identity quaternions - avoid in-place operations
    rotations_data = torch.zeros((num_points, 4), dtype=torch.float32, device=device)
    rotations_data[:, 0] = 1.0  # w component
    rotations = torch.nn.Parameter(rotations_data)
    
    # Initialize colors from COLMAP (if available) or random
    if points3d.shape[1] >= 6:  # Has color info
        colors = torch.tensor(points3d[:, 3:6], dtype=torch.float32, device=device, requires_grad=True)
    else:
        colors = torch.rand((num_points, 3), dtype=torch.float32, device=device, requires_grad=True)
    
    # Initialize opacities (1D shape for gsplat)
    opacities = torch.full((num_points,), 0.1, dtype=torch.float32, device=device, requires_grad=True)
    
    return positions, scales, rotations, colors, opacities

# Re-initialize the Gaussians
device = 'cuda' if torch.cuda.is_available() else 'cpu'
positions, scales, rotations, colors, opacities = initialize_gaussians(colmap_data['points3d'], device)
print(f" Initialized {len(positions)} Gaussians on {device}")
print(f"Opacities shape: {opacities.shape}")

✅ Initialized 5025 Gaussians on cuda
Opacities shape: torch.Size([5025])


In [25]:
# Cell 11: GSplat Training Loop (Fixed Matrix Formats)
def train_gaussians(colmap_data, positions, scales, rotations, colors, opacities, 
                   num_iterations=30000, lr=0.01, save_every=5000):
    """Train Gaussian Splatting model using gsplat."""
    
    # Move data to GPU
    images = [img.to(device) for img in colmap_data['images']]
    viewmats = [vm.to(device) for vm in colmap_data['viewmats']]
    Ks = [k.to(device) for k in colmap_data['Ks']]
    
    # Optimizer
    optimizer = torch.optim.Adam([positions, scales, rotations, colors, opacities], lr=lr)
    
    print(" Starting GSplat training...")
    
    for iteration in range(num_iterations):
        # Sample random camera view
        cam_idx = np.random.randint(len(images))
        gt_image = images[cam_idx]
        viewmat = viewmats[cam_idx]
        K = Ks[cam_idx]
        
        try:
            rendered_image, _, _ = gsplat.rasterization(
                means=positions,
                quats=rotations / torch.norm(rotations, dim=-1, keepdim=True),
                scales=torch.exp(scales),
                colors=torch.sigmoid(colors),
                opacities=torch.sigmoid(opacities),
                viewmats=viewmat.unsqueeze(0),  # [1, 4, 4]
                Ks=K.unsqueeze(0),  # [1, 3, 3] - intrinsic matrix
                width=colmap_data['width'],
                height=colmap_data['height']
            )
            
            rendered_image = rendered_image.squeeze(0).permute(1, 2, 0)
            loss = torch.nn.functional.mse_loss(rendered_image, gt_image)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if iteration % 100 == 0:
                print(f" Iteration {iteration}/{num_iterations} | Loss: {loss.item():.6f}")
            
            if iteration % save_every == 0 and iteration > 0:
                print(f" Checkpoint saved at iteration {iteration}")
                
        except Exception as e:
            print(f" Error at iteration {iteration}: {str(e)[:100]}...")
            if iteration > 100:
                break
    
    print(" GSplat training complete!")
    return positions, scales, rotations, colors, opacities

# Start training
trained_positions, trained_scales, trained_rotations, trained_colors, trained_opacities = train_gaussians(
    colmap_data, positions, scales, rotations, colors, opacities, 
    num_iterations=30000, lr=0.01
)

🚀 Starting GSplat training...


/opt/conda/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not
set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(

❌ Error at iteration 0: Error building extension 'gsplat_cuda': [1/30] /usr/local/cuda/bin/nvcc --generate-dependencies-with...


❌ Error at iteration 1: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


❌ Error at iteration 2: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


...

❌ Error at iteration 97: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


❌ Error at iteration 98: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


❌ Error at iteration 99: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


❌ Error at iteration 100: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...


❌ Error at iteration 101: /home/jovyan/.cache/torch_extensions/py312_cu128/gsplat_cuda/gsplat_cuda.so: cannot open shared obje...
✅ GSplat training complete!


In [ ]:
# Cell 12: Save functions for PLY and SPLAT formats
def save_gaussians_ply(positions, scales, rotations, colors, opacities, output_path):
    """Save Gaussians in PLY format (compatible with original Gaussian Splatting)."""
    
    # Convert tensors to numpy and move to CPU
    pos = positions.detach().cpu().numpy()
    scale = scales.detach().cpu().numpy()
    rot = rotations.detach().cpu().numpy()
    color = colors.detach().cpu().numpy()
    opacity = opacities.detach().cpu().numpy()
    
    # Normalize quaternions
    rot = rot / np.linalg.norm(rot, axis=1, keepdims=True)
    
    # Apply activations
    scale = np.exp(scale)
    color = 1 / (1 + np.exp(-color))  # sigmoid
    opacity = 1 / (1 + np.exp(-opacity))  # sigmoid
    
    # Convert colors to spherical harmonics (DC component only)
    SH_C0 = 0.28209479177387814
    sh_dc = (color - 0.5) / SH_C0
    
    # Create PLY data
    num_points = len(pos)
    
    # PLY header
    ply_header = f"""ply
format binary_little_endian 1.0
element vertex {num_points}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
property float opacity
property float scale_0
property float scale_1
property float scale_2
property float rot_0
property float rot_1
property float rot_2
property float rot_3
property float f_dc_0
property float f_dc_1
property float f_dc_2
end_header
"""
    
    with open(output_path, 'wb') as f:
        f.write(ply_header.encode())
        
        for i in range(num_points):
            # Position
            f.write(struct.pack('fff', pos[i, 0], pos[i, 1], pos[i, 2]))
            # Color (as uint8)
            f.write(struct.pack('BBB', 
                               int(color[i, 0] * 255),
                               int(color[i, 1] * 255), 
                               int(color[i, 2] * 255)))
            # Opacity
            f.write(struct.pack('f', opacity[i, 0]))
            # Scales (log space)
            f.write(struct.pack('fff', 
                               np.log(scale[i, 0]),
                               np.log(scale[i, 1]), 
                               np.log(scale[i, 2])))
            # Rotations (quaternion)
            f.write(struct.pack('ffff', rot[i, 0], rot[i, 1], rot[i, 2], rot[i, 3]))
            # Spherical harmonics DC component
            f.write(struct.pack('fff', sh_dc[i, 0], sh_dc[i, 1], sh_dc[i, 2]))

def save_gaussians_splat(positions, scales, rotations, colors, opacities, output_path):
    """Save Gaussians in SPLAT format for web viewers."""
    
    # Convert tensors to numpy and move to CPU
    pos = positions.detach().cpu().numpy()
    scale = scales.detach().cpu().numpy()
    rot = rotations.detach().cpu().numpy()
    color = colors.detach().cpu().numpy()
    opacity = opacities.detach().cpu().numpy()
    
    # Apply activations
    scale = np.exp(scale)
    color = 1 / (1 + np.exp(-color))  # sigmoid
    opacity = 1 / (1 + np.exp(-opacity))  # sigmoid
    
    # Normalize quaternions
    rot = rot / np.linalg.norm(rot, axis=1, keepdims=True)
    
    # Sort by size (largest first)
    sizes = np.prod(scale, axis=1)
    sorted_indices = np.argsort(-sizes)
    
    with open(output_path, 'wb') as f:
        for idx in sorted_indices:
            # Position (3 floats)
            f.write(pos[idx].astype(np.float32).tobytes())
            # Scale (3 floats)
            f.write(scale[idx].astype(np.float32).tobytes())
            # Color + opacity (4 uint8)
            rgba = np.concatenate([color[idx], opacity[idx]])
            f.write((rgba * 255).clip(0, 255).astype(np.uint8).tobytes())
            # Rotation quaternion (4 uint8, normalized and shifted)
            rot_uint8 = ((rot[idx] * 128) + 128).clip(0, 255).astype(np.uint8)
            f.write(rot_uint8.tobytes())

In [ ]:
# Cell 13: Save final outputs
export_name = f"{run_uuid}.ply"
splat_name = f"{run_uuid}.splat"

# Save PLY format
ply_path = gsplat_dir / export_name
save_gaussians_ply(trained_positions, trained_scales, trained_rotations, 
                  trained_colors, trained_opacities, ply_path)
print(f"✅ PLY file saved to: {ply_path}")

# Save SPLAT format  
splat_path = gsplat_dir / splat_name
save_gaussians_splat(trained_positions, trained_scales, trained_rotations,
                    trained_colors, trained_opacities, splat_path)
print(f"✅ SPLAT file saved to: {splat_path}")

print(f"""
🎉 **GSplat Training Complete!**

📁 **Output Files:**
- PLY: `{ply_path}`
- SPLAT: `{splat_path}`

📊 **Training Stats:**
- Total Gaussians: {len(trained_positions)}
- Training Images: {len(colmap_data['images'])}
- Final Loss: Check last iteration above

🚀 **Next Steps:**
- View your .splat file in a web viewer
- Use the .ply file with other Gaussian Splatting tools
- Try different hyperparameters for better quality
""")